# Macro exploration (scratch)

This notebook assumes the working directory is the repo root.

Files:
- `data/processed/macro_timeseries_wide.xlsx` (wide time series, one col per FRED `series_id`)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="darkgrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

In [ ]:
processed_dir = Path("data/processed")
if not processed_dir.exists():
    processed_dir = Path("../../../data/processed")

macro_df = pd.read_excel(
    processed_dir / "macro_timeseries_wide.xlsx",
    sheet_name="macro",
)

display(macro_df.head())

## Wide macro timeseries: `macro_timeseries_wide.xlsx`


In [ ]:
import importlib
import sys

repo_root = processed_dir.parent.parent
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

config = importlib.import_module("src.config")

macro_df["date"] = pd.to_datetime(macro_df["date"], errors="coerce")
macro_df = macro_df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

series_cols = [c for c in macro_df.columns if c != "date"]
for col in series_cols:
    macro_df[col] = pd.to_numeric(macro_df[col], errors="coerce")

series_rows = []
for series_id in series_cols:
    s = macro_df[series_id]
    dates = macro_df.loc[s.notna(), "date"]
    deltas = dates.sort_values().diff().dropna()
    median_spacing_days = None
    if not deltas.empty:
        median_spacing_days = float(deltas.median() / pd.Timedelta(days=1))

    series_rows.append(
        {
            "series_id": series_id,
            "series_name": config.FRED_DEFAULT_SERIES.get(series_id, ""),
            "dtype": str(s.dtype),
            "n_obs": int(s.notna().sum()),
            "missing_pct": float(s.isna().mean() * 100),
            "start": dates.min(),
            "end": dates.max(),
            "median_spacing_days": median_spacing_days,
        }
    )

series_summary = (
    pd.DataFrame(series_rows)
    .set_index("series_id")
    .sort_values(["missing_pct", "n_obs"], ascending=[False, True])
)

print("shape:", macro_df.shape)
series_summary.round({"missing_pct": 2, "median_spacing_days": 2}).head(20)

In [ ]:
top_missing = (
    series_summary.sort_values("missing_pct", ascending=False)
    .head(15)
    .sort_values("missing_pct")
)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_missing["missing_pct"], y=top_missing.index, color="#4C72B0")
plt.title("Series: missing % (expected w/ mixed frequencies)")
plt.xlabel("Missing %")
plt.ylabel("series_id")
plt.tight_layout()
plt.show()

In [ ]:
by_obs = series_summary.sort_values("n_obs")

plt.figure(figsize=(10, 7))
sns.barplot(x=by_obs["n_obs"], y=by_obs.index, color="#55A868")
plt.title("Series: number of observations")
plt.xlabel("observations")
plt.ylabel("series_id")
plt.tight_layout()
plt.show()

In [ ]:
coverage = macro_df[series_cols].notna().sum(axis=1)

plt.figure(figsize=(12, 4))
sns.lineplot(x=macro_df["date"], y=coverage, color="#C44E52")
plt.title("Wide dataset coverage: # observed series per date")
plt.xlabel("date")
plt.ylabel("# series with a value")
plt.tight_layout()
plt.show()

## Selected series plots


In [ ]:
plot_series = ["GDPC1", "CPIAUCSL", "UNRATE", "FEDFUNDS", "DGS10", "SP500"]

fig, axes = plt.subplots(
    len(plot_series),
    1,
    figsize=(12, 2.5 * len(plot_series)),
    sharex=True,
)

for ax, series_id in zip(axes, plot_series):
    if series_id not in macro_df.columns:
        ax.set_visible(False)
        continue

    s_df = macro_df[["date", series_id]].dropna()
    sns.lineplot(data=s_df, x="date", y=series_id, ax=ax, color="#4C72B0")

    title = f"{series_id}"
    series_name = config.FRED_DEFAULT_SERIES.get(series_id, "")
    if series_name:
        title += f" — {series_name}"
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("value")

axes[-1].set_xlabel("date")
plt.tight_layout()
plt.show()

In [ ]:
dist_series = ["UNRATE", "FEDFUNDS", "DGS10", "T10Y2Y", "SP500", "VIXCLS"]

for series_id in dist_series:
    if series_id not in macro_df.columns:
        continue

    plt.figure(figsize=(10, 4))
    sns.histplot(macro_df[series_id].dropna(), bins=40, kde=True, color="#8172B3")

    title = f"Distribution: {series_id}"
    series_name = config.FRED_DEFAULT_SERIES.get(series_id, "")
    if series_name:
        title += f" — {series_name}"
    plt.title(title)
    plt.xlabel(series_id)
    plt.tight_layout()
    plt.show()

In [ ]:
monthly_df = macro_df.set_index("date")[series_cols].resample("ME").last().ffill()
monthly_diff = monthly_df.diff()
corr = monthly_diff.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap="vlag", center=0, vmin=-1, vmax=1)
plt.title("Monthly first-difference correlation (resample M + ffill)")
plt.tight_layout()
plt.show()